# Llama 3.2 1B Instruct: DPO with LoRA + AdamW

This recipe starts from `meta-llama/Llama-3.2-1B-Instruct`, uses the model's native chat template, trains DPO through LoRA adapters, and uses AdamW. Because LoRA has much lower trainable capacity than full fine-tuning, the default run uses 8k preference pairs rather than the 20k full-DPO recipe. The frozen base and reference can stay in fp16; only the LoRA adapters are trainable fp32 parameters.

In [ ]:
from pathlib import Path
import os

# Kaggle: add HF_TOKEN under Add-ons > Secrets. Meta Llama models are gated,
# so your Hugging Face account must have access to the selected model.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception as exc:
    print(f"No Kaggle HF_TOKEN secret loaded: {exc}")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])

PROJECT_DIR = Path("/kaggle/working/qwen-dpo-alignment")
REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
# Kaggle currently preinstalls an old torchao that PEFT refuses to import.
# We are not using torchao quantization for this LoRA run, so remove it.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e . --no-deps

In [ ]:
from pathlib import Path

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
PROMPT_FORMAT = "chat_template"
DATASET_DPO = "HuggingFaceH4/ultrafeedback_binarized"

RUN_ROOT = Path("outputs/llama32_1b_instruct_dpo_lora_adamw")
DPO_OUT = RUN_ROOT / "dpo"

DPO_TRAIN_EXAMPLES = 8000
DPO_VAL_EXAMPLES = 500
MAX_LENGTH = 384
MAX_PROMPT_LENGTH = 192
GRAD_ACCUM = 32
EVAL_STEPS = 32
SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"

## Memory Estimate

This is an estimate, not a CUDA allocator trace. It separates frozen fp16 model weights from LoRA AdamW state so we can reason about whether AdamW is viable under mixed precision.

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(MODEL_NAME, token=os.environ.get("HF_TOKEN"))
hidden = cfg.hidden_size
intermediate = cfg.intermediate_size
layers = cfg.num_hidden_layers
vocab = cfg.vocab_size
rank = LORA_R

# For Llama-style blocks with q/k/v/o and gate/up/down adapted:
# attention LoRA params per layer: 4 * r * (hidden + hidden)
# MLP LoRA params per layer: 3 * r * (hidden + intermediate)
lora_params = layers * rank * (8 * hidden + 3 * (hidden + intermediate))

# Conservative approximate base params from common decoder-only components.
# This is an estimate; the exact value prints during model load if needed.
base_params_est = (
    vocab * hidden +
    layers * (
        4 * hidden * hidden +
        3 * hidden * intermediate +
        2 * hidden
    ) +
    hidden
)

policy_base_gb = base_params_est * 2 / 1024**3          # fp16 frozen base
reference_gb = base_params_est * 2 / 1024**3            # fp16 frozen reference
lora_adamw_gb = lora_params * (4 + 4 + 8) / 1024**3     # fp32 params + grads + AdamW m/v

print(f"Config: layers={layers}, hidden={hidden}, intermediate={intermediate}, vocab={vocab}")
print(f"Estimated base params: {base_params_est/1e9:.2f}B")
print(f"Estimated LoRA trainable params: {lora_params/1e6:.1f}M")
print(f"Policy frozen fp16 base: {policy_base_gb:.2f} GB")
print(f"Reference frozen fp16 base: {reference_gb:.2f} GB")
print(f"LoRA AdamW train state: {lora_adamw_gb:.2f} GB")
print(f"Weights + optimizer only, one GPU: {policy_base_gb + reference_gb + lora_adamw_gb:.2f} GB")
print("Add activation/logit/cuda overhead. With MAX_LENGTH=384 and batch=1, this is usually plausible on 16GB; two GPUs are safer because the script places the reference on cuda:1 when available.")

## DPO LoRA Training

With LoRA, AdamW becomes realistic because the optimizer state is attached only to adapter weights, not the full 1B-parameter model. The reference model is still frozen and loaded in fp16; if Kaggle gives two GPUs, the script puts it on `cuda:1`.

In [ ]:
!python scripts/train_dpo.py \
  --model_name_or_path {MODEL_NAME} \
  --dataset_name {DATASET_DPO} \
  --train_split train \
  --val_split test \
  --output_dir {DPO_OUT} \
  --max_train_examples {DPO_TRAIN_EXAMPLES} \
  --max_val_examples {DPO_VAL_EXAMPLES} \
  --max_length {MAX_LENGTH} \
  --max_prompt_length {MAX_PROMPT_LENGTH} \
  --prompt_format {PROMPT_FORMAT} \
  --per_device_batch_size 1 \
  --gradient_accumulation_steps {GRAD_ACCUM} \
  --num_epochs 1 \
  --learning_rate 1e-4 \
  --optimizer adamw \
  --weight_decay 0.0 \
  --beta 0.1 \
  --use_lora \
  --lora_r {LORA_R} \
  --lora_alpha {LORA_ALPHA} \
  --lora_dropout {LORA_DROPOUT} \
  --lora_target_modules {LORA_TARGET_MODULES} \
  --logging_steps 5 \
  --eval_steps {EVAL_STEPS} \
  --seed {SEED}

## Training Curves

In [ ]:
!python scripts/plot_training_curves.py \
  --metrics {DPO_OUT}/metrics.jsonl \
  --output_dir {RUN_ROOT}/plots \
  --prefix llama32_1b_instruct_dpo_lora_adamw

## Optional Smoke Generation

In [ ]:
# Optional quick generation smoke test. The saved LoRA adapter path can now be loaded directly.
EVAL_DIR = RUN_ROOT / "alpaca_smoke"
!python scripts/make_alpaca_subset.py --size 20 --seed {SEED} --output_dir {EVAL_DIR}
!python scripts/generate_alpaca.py \
  --model_name_or_path {DPO_OUT}/final \
  --subset_path {EVAL_DIR}/alpacaeval_20_seed{SEED}.json \
  --output_path {EVAL_DIR}/llama32_1b_instruct_dpo_lora_outputs.json \
  --generator_name llama32_1b_instruct_dpo_lora_adamw \
  --prompt_format chat_template \
  --batch_size 4 \
  --max_new_tokens 256 \
  --temperature 0.0